# Spendly — Isolation Forest finalization candidate

Builds on the supplied V2.1 behavioural feature design. This is a **development
comparison and freeze workflow**, not a claim of final-test superiority.

## Run on Kaggle or Colab (CPU)
The verified Windows run used Python3.10.6, numpy1.26.4, pandas2.3.3,
scikit-learn1.7.2 and joblib1.6.0 (see if_final_requirements.txt).
No TensorFlow, GPU or downloads in code. Other runtimes must record their versions;
cross-platform byte-identical artifacts are not promised.
Attach original R3 manifest.json and development train.zip, calibration.zip,
validation.zip. They may be together or ZIPs in a development/ subfolder.
Set INPUT_ROOT to that directory. Do not attach final/shifted test files.

Both estimators fit identical mixed-label training rows before2023-04-24.
Calibration users during2023-04-24–2023-08-28 set a negative-only1% FPR cutoff.
Validation users during2023-08-28–2024-01-01 provide the later comparison.
Past-only user history is available for features; evaluation labels never choose
the threshold. The reference is the preserved V6 excess feature view under this
new common protocol, not a reproduction of its old metric table.

The candidate uses Nairobi time, normalized nanoseconds and simultaneous-batch
exclusion. Predictions are saved for auditing and paired user-bootstrap intervals.
These are reused synthetic development cohorts, not new independent evidence.


In [ ]:
from pathlib import Path
import hashlib, sys
MODULES = {'if_final_features.py': '"""Causal V2.1-derived behavioural features; no labels required for inference.\n\nEqual user timestamps are a batch: none of its transactions can see another.\nTimestamps must carry an offset; behavioural time is Africa/Nairobi.\n"""\nimport numpy as np\nimport pandas as pd\n\nFEATURES = [\n    \'log_amount\', \'hour_sin\', \'hour_cos\', \'dow_sin\', \'dow_cos\', \'is_weekend\',\n    \'user_z\', \'user_ratio\', \'category_z\', \'category_ratio\', \'merchant_z\', \'merchant_ratio\',\n    \'positive_category_increase_ratio\', \'log_gap_user_minutes\', \'log_gap_category_minutes\',\n    \'log_gap_merchant_minutes\', \'log_gap_same_amount_minutes\',\n    \'log_gap_same_merchant_amount_minutes\', \'log_gap_same_category_amount_minutes\',\n    \'same_amount_within_1h\', \'same_amount_within_7d\', \'same_merchant_amount_within_1h\',\n    \'same_category_amount_within_1h\', \'same_category_amount_within_7d\',\n    \'ratio_previous_category\', \'absolute_log_category_change\',\n    \'log_user_txn_count_prev_1h\', \'log_user_txn_count_prev_24h\',\n    \'log_category_txn_count_prev_7d\', \'is_recurring_category\', \'recurring_amount_change\',\n    \'recurring_increase_ratio\', \'log_recurring_gap_days\', \'category_prior_share\',\n    \'merchant_prior_share\', \'log_history_days\', \'amount_vs_user_recent_median_50\',\n    \'amount_vs_user_recent_q90_50\', \'amount_vs_category_recent_median_20\',\n    \'amount_vs_category_recent_q90_20\',\n]\n\n\ndef aware_dates(values):\n    """Reject ambiguous naive timestamps, normalize unit and Nairobi timezone."""\n    values = pd.Series(values)\n    if isinstance(values.dtype, pd.DatetimeTZDtype):\n        dates = pd.to_datetime(values, utc=True)\n    else:\n        text = values.astype(str)\n        if not text.str.contains(r\'(?:Z|[+-]\\d{2}:?\\d{2})$\', regex=True).all():\n            raise ValueError(\'Every timestamp must include a timezone offset\')\n        dates = pd.to_datetime(values, utc=True, format=\'mixed\', errors=\'raise\')\n    if dates.isna().any():\n        raise ValueError(\'Missing timestamp\')\n    return dates.astype(\'datetime64[ns, UTC]\').dt.tz_convert(\'Africa/Nairobi\')\n\n\ndef validate_transactions(raw):\n    required = {\'transaction_id\', \'user_id\', \'transaction_timestamp\', \'amount\',\n                \'category\', \'merchant\', \'transaction_type\'}\n    if not required <= set(raw):\n        raise ValueError(f\'Missing columns: {sorted(required-set(raw))}\')\n    x = raw.copy()\n    for name in (\'transaction_id\', \'user_id\', \'category\', \'merchant\', \'transaction_type\'):\n        if x[name].isna().any() or x[name].astype(str).str.strip().eq(\'\').any():\n            raise ValueError(f\'Missing {name}\')\n        x[name] = x[name].astype(str)\n    if x.transaction_id.duplicated().any():\n        raise ValueError(\'Duplicate transaction IDs\')\n    if not x.transaction_type.isin([\'expense\', \'income\']).all():\n        raise ValueError(\'Unsupported transaction type\')\n    x[\'amount\'] = pd.to_numeric(x.amount, errors=\'raise\').astype(float)\n    if not np.isfinite(x.amount).all() or x.amount.le(0).any():\n        raise ValueError(\'Amounts must be finite and positive\')\n    x[\'transaction_timestamp\'] = aware_dates(x.transaction_timestamp)\n    return x.sort_values([\'user_id\', \'transaction_timestamp\', \'transaction_id\']).reset_index(drop=True)\n\n\ndef build_features(raw):\n    x = validate_transactions(raw)\n    x = x.loc[x.transaction_type.eq(\'expense\')].reset_index(drop=True)\n    out = x.copy()\n    amount = x.amount\n    stamp = x.transaction_timestamp\n    x[\'_square\'] = amount**2\n    x[\'_amount_key\'] = amount.round(2)\n\n    batch_indices = {}\n\n    def batch_prior(series, keys):\n        # Preserve NaNs: groupby.first() would skip them and leak another row.\n        key = tuple(keys)\n        if key not in batch_indices:\n            groupers = [x[k] for k in keys] + [stamp]\n            batch_indices[key] = pd.Series(np.arange(len(x))).groupby(groupers, sort=False).transform(\'min\').to_numpy()\n        return series.iloc[batch_indices[key]].reset_index(drop=True)\n\n    def previous_count(keys):\n        return batch_prior(x.groupby(keys, sort=False).cumcount().astype(float), keys)\n\n    for prefix, keys in [(\'user\', [\'user_id\']), (\'category\', [\'user_id\', \'category\']),\n                         (\'merchant\', [\'user_id\', \'merchant\'])]:\n        group = x.groupby(keys, sort=False)\n        count = previous_count(keys)\n        total = batch_prior(group.amount.cumsum()-amount, keys)\n        square = batch_prior(group[\'_square\'].cumsum()-x[\'_square\'], keys)\n        mean = total/count.replace(0, np.nan)\n        variance = (square-total**2/count.replace(0, np.nan))/(count-1).where(count.gt(1))\n        out[prefix+\'_z\'] = ((amount-mean)/(np.sqrt(variance.clip(lower=0))+1e-6)).clip(-20, 20)\n        out[prefix+\'_ratio\'] = (amount/(mean+1)).clip(0, 50)\n\n    prior_category = batch_prior(x.groupby([\'user_id\', \'category\']).amount.shift(), [\'user_id\', \'category\'])\n    ratio = (amount+1)/(prior_category+1)\n    out[\'ratio_previous_category\'] = ratio.clip(0, 50)\n    out[\'absolute_log_category_change\'] = np.abs(np.log(ratio.clip(1e-4, 100))).clip(0, 5)\n    out[\'positive_category_increase_ratio\'] = (ratio-1).clip(0, 10)\n    for prefix, keys, window, minimum in [(\'user\', [\'user_id\'], 50, 10),\n                                           (\'category\', [\'user_id\', \'category\'], 20, 5)]:\n        group = x.groupby(keys, sort=False).amount\n        median = group.transform(lambda s: s.shift().rolling(window, min_periods=minimum).median())\n        q90 = group.transform(lambda s: s.shift().rolling(window, min_periods=minimum).quantile(.9))\n        for name, reference in [(\'median\', median), (\'q90\', q90)]:\n            out[f\'amount_vs_{prefix}_recent_{name}_{window}\'] = (amount/(batch_prior(reference, keys)+1)).clip(0, 50)\n\n    def gap(keys, name):\n        prior = batch_prior(x.groupby(keys, sort=False).transaction_timestamp.shift(), keys)\n        minutes = (stamp-prior).dt.total_seconds()/60\n        out[f\'log_gap_{name}_minutes\'] = np.log1p(minutes.clip(lower=0))\n        return minutes\n\n    gap([\'user_id\'], \'user\')\n    category_gap = gap([\'user_id\', \'category\'], \'category\')\n    gap([\'user_id\', \'merchant\'], \'merchant\')\n    for name, keys in [(\'same_amount\', [\'user_id\', \'_amount_key\']),\n                       (\'same_merchant_amount\', [\'user_id\', \'merchant\', \'_amount_key\']),\n                       (\'same_category_amount\', [\'user_id\', \'category\', \'_amount_key\'])]:\n        minutes = gap(keys, name)\n        out[name+\'_within_1h\'] = minutes.le(60).astype(int)\n        if name != \'same_merchant_amount\':\n            out[name+\'_within_7d\'] = minutes.le(7*24*60).astype(int)\n\n    # Explicit nanoseconds; counts exclude every transaction in the current batch.\n    ns = stamp.astype(\'datetime64[ns, Africa/Nairobi]\').astype(\'int64\').to_numpy()\n    for name, keys, minutes in [(\'user_txn_count_prev_1h\', [\'user_id\'], 60),\n                               (\'user_txn_count_prev_24h\', [\'user_id\'], 1440),\n                               (\'category_txn_count_prev_7d\', [\'user_id\', \'category\'], 10080)]:\n        counts = np.zeros(len(x))\n        for positions in x.groupby(keys, sort=False).indices.values():\n            t = ns[positions]\n            counts[positions] = np.searchsorted(t, t, side=\'left\')-np.searchsorted(t, t-minutes*60*10**9, side=\'left\')\n        out[\'log_\'+name] = np.log1p(counts)\n\n    hour = stamp.dt.hour+stamp.dt.minute/60\n    dow = stamp.dt.dayofweek\n    out[\'log_amount\'] = np.log1p(amount)\n    for name, values, period in [(\'hour\', hour, 24), (\'dow\', dow, 7)]:\n        out[name+\'_sin\'] = np.sin(2*np.pi*values/period)\n        out[name+\'_cos\'] = np.cos(2*np.pi*values/period)\n    out[\'is_weekend\'] = dow.ge(5).astype(int)\n    out[\'history_days\'] = (stamp-x.groupby(\'user_id\').transaction_timestamp.transform(\'min\')).dt.total_seconds()/86400\n    out[\'log_history_days\'] = np.log1p(out.history_days)\n    recurring = x.category.str.casefold().isin([\'rent\', \'utilities\', \'subscriptions\']).astype(int)\n    out[\'is_recurring_category\'] = recurring\n    out[\'recurring_amount_change\'] = out.absolute_log_category_change*recurring\n    out[\'recurring_increase_ratio\'] = out.positive_category_increase_ratio*recurring\n    out[\'log_recurring_gap_days\'] = np.log1p(category_gap/1440)*recurring\n    for key in (\'category\', \'merchant\'):\n        out[key+\'_prior_share\'] = previous_count([\'user_id\', key])/(previous_count([\'user_id\'])+1)\n    if np.isinf(out[FEATURES].to_numpy()).any():\n        raise ValueError(\'Non-finite feature overflow\')\n    return out\n', 'if_final_reference.py': '"""Preserved five-feature V6 excess view for a matched-protocol comparison.\n\nThe function below is copied verbatim from the published V7 WORKER_SOURCE.\nTests compare its source and output to that notebook; no new reference tuning.\n"""\nfrom collections import defaultdict, deque\nimport numpy as np\nimport pandas as pd\nfrom if_final_features import validate_transactions\n\nA_FEATURES = [\'amount_ratio\', \'category_ratio\', \'category_deviation\', \'merchant_ratio\', \'count_hour\', \'sum_hour_ratio\', \'count_day\', \'sum_week_ratio\', \'duplicate_hour\', \'merchants_hour\', \'night_rarity\', \'category_rarity\', \'user_cold\', \'merchant_missing\']\nREFERENCE_COLUMNS = [\'category_excess\', \'merchant_excess\', \'duplicate_hour\', \'rare_large\', \'burst_amount\']\n\n\ndef anomaly_features_fast(data):\n    """Same 18 V5 features; integer time windows avoid repeated Timedelta/Pandas work."""\n    values=[]\n    for _, group in data.groupby(\'user_id\',sort=False):\n        group=group.sort_values([\'transaction_timestamp\',\'transaction_id\'])\n        history=deque(maxlen=100)\n        categories=defaultdict(lambda:deque(maxlen=40))\n        merchants=defaultdict(lambda:deque(maxlen=20))\n        recent,category_counts,seen,nights=deque(),defaultdict(int),0,0\n        rows=list(group.itertuples())\n        times=group.transaction_timestamp.to_numpy(dtype=\'datetime64[ns]\').astype(\'int64\')\n        boundaries=np.r_[0,np.flatnonzero(np.diff(times))+1,len(times)]\n        hour_ns,day_ns=3600*10**9,86400*10**9\n        for left,right in zip(boundaries[:-1],boundaries[1:]):\n            stamp=times[left]\n            hour_of_day=rows[left].transaction_timestamp.hour\n            while recent and recent[0][0] < stamp-7*day_ns:\n                recent.popleft()\n            hour=[r for r in recent if r[0]>=stamp-hour_ns]\n            day=[r for r in recent if r[0]>=stamp-day_ns]\n            for row in rows[left:right]:\n                a,c,m=row.amount,row.category,row.merchant\n                typical=max(float(np.median(history)) if history else 1.,1.)\n                cat=categories[c]\n                center=max(float(np.median(cat)) if cat else typical,1.)\n                spread=max(float(np.median(np.abs(np.asarray(cat)-center)))*1.4826 if cat else 0.,center*.2,1.)\n                mc=max(float(np.median(merchants[m])) if m and merchants[m] else center,1.)\n                feature=[a/typical,a/center,max(0.,(a-center)/spread),a/mc,len(hour),sum(r[1] for r in hour)/typical,\n                    len(day),sum(r[1] for r in recent)/typical,\n                    sum(bool(m) and r[2]==m and r[3]==c and abs(r[1]-a)<.01 for r in hour),\n                    len({r[2] for r in hour if r[2]}),float(hour_of_day<5 or hour_of_day>=23)*(1-(nights+1)/(seen+2)),\n                    1-category_counts[c]/max(1,seen),float(seen<10),float(not m)]\n                values.append((row.Index,feature))\n            for row in rows[left:right]:\n                history.append(row.amount); categories[row.category].append(row.amount)\n                if row.merchant:\n                    merchants[row.merchant].append(row.amount)\n                recent.append((stamp,row.amount,row.merchant,row.category))\n                seen+=1; nights+=int(hour_of_day<5 or hour_of_day>=23); category_counts[row.category]+=1\n    x=pd.DataFrame([v for _,v in values],index=[i for i,_ in values],columns=A_FEATURES,dtype=float).reindex(data.index)\n    x[\'category_excess\']=np.maximum(0.,x.category_ratio-1.)\n    x[\'merchant_excess\']=np.maximum(0.,x.merchant_ratio-1.)\n    x[\'rare_large\']=x.category_rarity*np.log1p(x.category_excess)*(1-x.user_cold)\n    x[\'burst_amount\']=x.count_hour*np.log1p(x.amount_ratio)\n    return x\n\n\ndef reference_features(raw):\n    data = validate_transactions(raw)\n    data = data.loc[data.transaction_type.eq(\'expense\')].reset_index(drop=True)\n    data[\'transaction_timestamp\'] = data.transaction_timestamp.dt.tz_localize(None)\n    values = np.log1p(anomaly_features_fast(data)[REFERENCE_COLUMNS])\n    values.index = data.transaction_id\n    return values\n', 'if_final_pipeline.py': '"""Development-only IF finalization, verified inputs and trusted-local bundles.\n\nDoes not expose a final-holdout loader. A separate approved final evaluation must\nbe predeclared after development review. Pickle bundles are trusted-local only.\n"""\nimport hashlib\nimport importlib.metadata\nimport io\nimport json\nfrom pathlib import Path\nimport platform\nimport zipfile\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import IsolationForest\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score\n\nfrom if_final_features import FEATURES, aware_dates, build_features, validate_transactions\n\nPROTOCOL = {\n    \'version\': \'if-finalization-v1\', \'dataset\': \'spendly-synthetic-r3-v1\',\n    \'training_end\': \'2023-04-24T00:00:00+03:00\',\n    \'calibration_end\': \'2023-08-28T00:00:00+03:00\',\n    \'evaluation_end\': \'2024-01-01T00:00:00+03:00\',\n    \'minimum_history_days\': 7, \'fpr_budget\': .01, \'seed\': 42,\n    \'fit_policy\': \'mixed training expenses; no normal-label filtering\',\n    \'threshold_policy\': \'negative-only calibration quantile, no F1 optimization\',\n    \'tie_policy\': \'candidate same-user timestamp batch excluded from own history\',\n    \'candidate\': {\'n_estimators\': 500, \'max_samples\': 16384, \'max_features\': .5},\n    \'reference\': {\'n_estimators\': 300, \'max_samples\': 128, \'max_features\': 1.},\n    \'selection\': \'no automatic promotion; paired development report for review\',\n    \'holdouts\': \'not accessed; finalization candidate only\',\n}\nCOHORTS = (\'train\', \'calibration\', \'validation\')\n\n\ndef digest_bytes(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    with path.open(\'x\', encoding=\'utf-8\') as stream:\n        json.dump(value, stream, indent=2, allow_nan=False)\n\n\ndef versions():\n    return {\'python\': platform.python_version(), **{name: importlib.metadata.version(name)\n            for name in (\'numpy\', \'pandas\', \'scikit-learn\', \'joblib\')}}\n\n\ndef source_identity():\n    root = Path(__file__).parent\n    return {name: digest_bytes((root/name).read_bytes()) for name in\n            (\'if_final_features.py\', \'if_final_pipeline.py\', \'if_final_reference.py\')}\n\n\ndef load_development(root):\n    """Only named development ZIPs are opened, never archive paths extracted."""\n    root = Path(root)\n    manifest_raw = (root/\'manifest.json\').read_bytes()\n    manifest = json.loads(manifest_raw)\n    if (manifest.get(\'version\') != PROTOCOL[\'dataset\'] or\n            manifest.get(\'synthetic_only\') is not True or manifest.get(\'weeks_per_user\') != 156):\n        raise ValueError(\'Expected original synthetic R3 manifest\')\n    data, identities, users, ids = {}, {}, set(), set()\n    for name in COHORTS:\n        entry = manifest[\'cohorts\'][name]\n        candidates = [p for p in (root/\'development\'/f\'{name}.zip\', root/f\'{name}.zip\') if p.is_file()]\n        if len(candidates) != 1:\n            raise ValueError(f\'Provide exactly one {name}.zip, at root or development/\')\n        packed = candidates[0].read_bytes()\n        if digest_bytes(packed) != entry[\'zip_sha256\']:\n            raise ValueError(f\'{name} ZIP hash mismatch\')\n        with zipfile.ZipFile(io.BytesIO(packed)) as archive:\n            files = [i for i in archive.infolist() if not i.is_dir()]\n            if len(files) != 1 or files[0].file_size != entry[\'csv_bytes\']:\n                raise ValueError(f\'{name} archive must contain exactly the declared CSV\')\n            raw = archive.read(files[0])\n        if digest_bytes(raw) != entry[\'sha256\']:\n            raise ValueError(f\'{name} CSV hash mismatch\')\n        frame = validate_transactions(pd.read_csv(io.BytesIO(raw)))\n        if len(frame) != entry[\'rows\'] or frame.user_id.nunique() != entry[\'users\']:\n            raise ValueError(f\'{name} counts mismatch\')\n        if not frame.is_anomaly.isin([0, 1]).all():\n            raise ValueError(\'Explicit binary labels required for evaluation\')\n        if users & set(frame.user_id) or ids & set(frame.transaction_id):\n            raise ValueError(\'Cohort identity overlap\')\n        users.update(frame.user_id); ids.update(frame.transaction_id)\n        start = aware_dates(frame.observation_start)\n        end = aware_dates(frame.observation_end)\n        if not start.eq(pd.Timestamp(\'2021-01-04T00:00:00+03:00\')).all() or not end.eq(pd.Timestamp(PROTOCOL[\'evaluation_end\'])).all():\n            raise ValueError(\'Unexpected R3 coverage\')\n        if not ((frame.transaction_timestamp >= start) & (frame.transaction_timestamp < end)).all():\n            raise ValueError(\'Transaction outside observation coverage\')\n        data[name] = frame\n        identities[name] = entry[\'sha256\']\n        print(f\'Verified {name}: {len(frame):,} rows / {entry["users"]} users\', flush=True)\n    return data, {\'manifest\': digest_bytes(manifest_raw), \'csv\': identities}\n\n\ndef capped_threshold(labels, scores, budget):\n    """Tied scores cannot exceed budget; positives never select the cutoff."""\n    labels, scores = np.asarray(labels), np.asarray(scores, dtype=float)\n    if not 0 < budget < 1 or len(labels) != len(scores) or not np.isfinite(scores).all():\n        raise ValueError(\'Invalid calibration\')\n    negative = np.sort(scores[labels == 0])[::-1]\n    if not len(negative):\n        raise ValueError(\'Calibration requires normal observations\')\n    allowed = int(np.floor(budget*len(negative)))\n    threshold = float(np.nextafter(negative[allowed], np.inf))\n    assert (negative >= threshold).sum() <= allowed\n    return threshold\n\n\ndef measures(rows, scores, threshold):\n    y = rows.is_anomaly.to_numpy(dtype=int)\n    scores = np.asarray(scores)\n    if len(y) != len(scores) or not len(y) or not np.isfinite(scores).all():\n        raise ValueError(\'Invalid evaluation scores\')\n    flags = scores >= threshold\n    tn, fp, fn, tp = map(int, confusion_matrix(y, flags, labels=[0, 1]).ravel())\n    both = len(np.unique(y)) == 2\n    precision = tp/(tp+fp) if tp+fp else 0.\n    recall = tp/(tp+fn) if tp+fn else None\n    fpr = fp/(fp+tn) if fp+tn else None\n    families = {}\n    for name, group in rows.assign(flag=flags).loc[rows.is_anomaly.eq(1)].groupby(\'anomaly_type\'):\n        families[str(name)] = {\'support\': len(group), \'detected\': int(group.flag.sum()), \'recall\': float(group.flag.mean())}\n    positive = rows.assign(flag=flags).loc[rows.is_anomaly.eq(1)]\n    events = positive.dropna(subset=[\'event_id\']).groupby([\'user_id\', \'event_id\']).flag.max()\n    days = pd.DataFrame({\'user\': rows.user_id, \'date\': rows.transaction_timestamp.dt.date, \'flag\': flags}).groupby([\'user\', \'date\']).flag.sum()\n    return {\'rows\': len(rows), \'TP\': tp, \'FP\': fp, \'FN\': fn, \'TN\': tn,\n            \'precision\': precision, \'recall\': recall, \'F1\': 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.,\n            \'FPR\': fpr, \'accuracy\': (tp+tn)/len(y),\n            \'balanced_accuracy\': (recall+1-fpr)/2 if both else None,\n            \'AP\': float(average_precision_score(y, scores)) if both else None,\n            \'ROC_AUC\': float(roc_auc_score(y, scores)) if both else None,\n            \'alert_rate\': float(flags.mean()), \'families\': families,\n            \'event_support\': len(events), \'event_recall\': float(events.mean()) if len(events) else None,\n            \'positive_rows_without_event_id\': int(positive.event_id.isna().sum()),\n            \'active_user_days\': len(days), \'alerted_user_days\': int(days.gt(0).sum()),\n            \'max_alerts_per_active_user_day\': int(days.max())}\n\n\ndef paired_interval(rows, candidate, reference, repetitions=1000):\n    """Paired user-cluster bootstrap; reused-development uncertainty only."""\n    labels = rows.is_anomaly.to_numpy(dtype=bool)\n    columns = []\n    for flags in (candidate, reference):\n        columns.extend([labels & flags, ~labels & flags, labels & ~flags])\n    counts = pd.DataFrame(np.array(columns).T.astype(int)).assign(user=rows.user_id.to_numpy()).groupby(\'user\').sum().to_numpy()\n    rng = np.random.default_rng(42)\n    differences = []\n    for _ in range(repetitions):\n        summed = counts[rng.integers(0, len(counts), size=len(counts))].sum(axis=0)\n        f1 = lambda v: 2*v[0]/max(1, 2*v[0]+v[1]+v[2])\n        differences.append(f1(summed[:3])-f1(summed[3:]))\n    return {\'method\': \'paired user-cluster bootstrap\', \'replicates\': repetitions,\n            \'candidate_minus_reference_F1_95pct\': np.quantile(differences, [.025, .975]).tolist()}\n\n\ndef save_bundle(directory, pipeline, columns, threshold, identity, probe):\n    directory = Path(directory)\n    directory.mkdir(exist_ok=False)\n    target = directory/\'pipeline.joblib\'\n    joblib.dump(pipeline, target)\n    manifest = {\'protocol\': PROTOCOL, \'versions\': versions(), \'sources\': source_identity(),\n                \'inputs\': identity, \'features\': columns, \'threshold\': threshold,\n                \'sha256\': digest_bytes(target.read_bytes()), \'status\': \'development_candidate_not_promoted\'}\n    write_json(directory/\'manifest.json\', manifest)\n    loaded, metadata = load_bundle(directory, trusted=True)\n    before = -pipeline.score_samples(probe)\n    after = -loaded.score_samples(probe)\n    np.testing.assert_array_equal(before, after)\n    write_json(directory/\'reload_parity.json\', {\'rows\': len(probe), \'max_abs_error\': float(np.max(np.abs(before-after))),\n                                             \'model_sha256\': metadata[\'sha256\']})\n\n\ndef load_bundle(directory, *, trusted=False):\n    if not trusted:\n        raise PermissionError(\'Only load your own trusted local joblib artifact\')\n    directory = Path(directory)\n    meta = json.loads((directory/\'manifest.json\').read_text())\n    if meta[\'versions\'] != versions() or meta[\'sources\'] != source_identity() or meta[\'protocol\'] != PROTOCOL:\n        raise ValueError(\'Frozen environment/source/protocol mismatch\')\n    path = directory/\'pipeline.joblib\'\n    if digest_bytes(path.read_bytes()) != meta[\'sha256\']:\n        raise ValueError(\'Model hash mismatch\')\n    return joblib.load(path), meta\n\n\ndef run_development(input_root, output):\n    from sklearn.pipeline import make_pipeline\n    from if_final_reference import reference_features\n    output = Path(output)\n    output.mkdir(exist_ok=False)\n    write_json(output/\'protocol.json\', {\'protocol\': PROTOCOL, \'sources\': source_identity(), \'versions\': versions()})\n    data, identity = load_development(input_root)\n    candidate, reference = {}, {}\n    for name, frame in data.items():\n        if name == \'train\':\n            frame = frame.loc[frame.transaction_timestamp < pd.Timestamp(PROTOCOL[\'training_end\'])].copy()\n        print(f\'Building {name} feature views\', flush=True)\n        candidate[name] = build_features(frame)\n        reference[name] = reference_features(frame)\n    # Common eligibility including reference warm-up, identical IDs in both views.\n    selected = {}\n    for name in COHORTS:\n        c = candidate[name]\n        mask = c.history_days.ge(PROTOCOL[\'minimum_history_days\'])\n        t = c.transaction_timestamp\n        if name == \'train\':\n            mask &= t.lt(pd.Timestamp(PROTOCOL[\'training_end\']))\n        elif name == \'calibration\':\n            mask &= t.ge(pd.Timestamp(PROTOCOL[\'training_end\'])) & t.lt(pd.Timestamp(PROTOCOL[\'calibration_end\']))\n        else:\n            mask &= t.ge(pd.Timestamp(PROTOCOL[\'calibration_end\'])) & t.lt(pd.Timestamp(PROTOCOL[\'evaluation_end\']))\n        shared = set(reference[name].index)\n        c = c.loc[mask & c.transaction_id.isin(shared)].set_index(\'transaction_id\')\n        if c.empty:\n            raise ValueError(f\'No common eligible {name} rows\')\n        selected[name] = c\n        reference[name] = reference[name].loc[c.index]\n    write_json(output/\'input_identity.json\', identity)\n    reports, flags, all_scores = {}, {}, {}\n    for name in (\'reference\', \'candidate\'):\n        matrices = ({k: v[FEATURES] for k, v in selected.items()} if name == \'candidate\' else reference)\n        params = PROTOCOL[name]\n        pipeline = make_pipeline(SimpleImputer(strategy=\'median\', add_indicator=True),\n            IsolationForest(**params, contamination=\'auto\', random_state=42, n_jobs=2)) if name == \'candidate\' else make_pipeline(\n                IsolationForest(**params, contamination=\'auto\', random_state=42, n_jobs=2))\n        print(f\'Fitting {name}: {len(matrices["train"]):,} common rows\', flush=True)\n        pipeline.fit(matrices[\'train\'])\n        calibration_scores = -pipeline.score_samples(matrices[\'calibration\'])\n        threshold = capped_threshold(selected[\'calibration\'].is_anomaly, calibration_scores, PROTOCOL[\'fpr_budget\'])\n        scores = -pipeline.score_samples(matrices[\'validation\'])\n        reports[name] = {\'threshold\': threshold, \'calibration\': measures(selected[\'calibration\'], calibration_scores, threshold),\n                         \'validation\': measures(selected[\'validation\'], scores, threshold)}\n        flags[name] = scores >= threshold\n        all_scores[name] = scores\n        save_bundle(output/name, pipeline, list(matrices[\'train\']), threshold, identity, matrices[\'validation\'].iloc[:128])\n        write_json(output/f\'{name}_metrics.json\', reports[name])\n    rows = selected[\'validation\']\n    prediction_rows = rows[[\'user_id\', \'transaction_timestamp\', \'is_anomaly\', \'anomaly_type\', \'event_id\']].copy()\n    for name in flags:\n        prediction_rows[name+\'_score\'] = all_scores[name]\n        prediction_rows[name+\'_flag\'] = flags[name]\n    prediction_rows.to_csv(output/\'development_predictions.csv\')\n    report = {\'scope\': \'reused synthetic development comparison; not final test\', \'protocol\': PROTOCOL,\n              \'metrics\': reports, \'uncertainty\': paired_interval(rows, flags[\'candidate\'], flags[\'reference\']),\n              \'decision\': \'review_required_no_automatic_promotion\',\n              \'common_rows\': {k: len(v) for k, v in selected.items()}}\n    write_json(output/\'comparison.json\', report)\n    write_json(output/\'artifacts.json\', {str(p.relative_to(output)): digest_bytes(p.read_bytes())\n                                      for p in output.rglob(\'*\') if p.is_file()})\n    print(json.dumps(report, indent=2))\n    return report\n'}
EXPECTED = {'if_final_features.py': '4694db7adb7ea73327f35d076fcfa0ebbe886572d90879ff2a8b65b811916821', 'if_final_reference.py': '46433927ac40249b2b08fd8f589bd183cc1c6dceb9ab8743e14281be8294bc26', 'if_final_pipeline.py': 'f8ec401b10d76d94d1029062ee82246c393667ca4ba351f1c14165bb40c1a860'}
MODULE_DIR = Path('/kaggle/working/if_finalization_modules') if Path('/kaggle').exists() else Path('if_finalization_modules')
MODULE_DIR.mkdir(exist_ok=True)
for name, source in MODULES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == EXPECTED[name]
    path = MODULE_DIR/name
    if path.exists() and path.read_text(encoding='utf-8') != source:
        raise RuntimeError('Existing module differs; use a new clean session/folder')
    path.write_text(source, encoding='utf-8')
sys.path.insert(0, str(MODULE_DIR.resolve()))
for name in ('if_final_features', 'if_final_reference', 'if_final_pipeline'):
    if name in sys.modules:
        raise RuntimeError('Restart kernel before rerunning setup to prevent stale imports')
from if_final_pipeline import PROTOCOL, run_development, versions
print(versions())
print(PROTOCOL)


In [ ]:
# Set paths explicitly. The output must not exist (protects prior runs).
INPUT_ROOT = Path('/kaggle/input/YOUR_R3_DATASET')
OUTPUT_ROOT = Path('/kaggle/working/if_finalization_run') if Path('/kaggle').exists() else Path('if_finalization_run')
RUN_DEVELOPMENT = False  # set True after checking the displayed protocol and paths
if RUN_DEVELOPMENT:
    report = run_development(INPUT_ROOT, OUTPUT_ROOT)
else:
    print('Configured only. Set INPUT_ROOT and RUN_DEVELOPMENT=True to run.')


## Preserve evidence and review
Download the entire run folder (or use the ZIP cell below). It includes the
protocol/source/environment identities, input hashes, both frozen pipelines,
reload parity, metrics, family/event burden, row scores and paired uncertainty.
Row predictions are synthetic audit evidence; do not commit them or model binaries.

Promotion is manual after reviewing gains, false-positive burden and family
trade-offs. No final-test reader is included: approve and lock that separate
evaluation only after this comparison. Do not tune using final-test outcomes.
Joblib is executable serialization: reload only your own trusted artifacts.


In [ ]:
import shutil
if RUN_DEVELOPMENT:
    archive = OUTPUT_ROOT.with_suffix('.zip')
    if archive.exists():
        raise FileExistsError(archive)
    shutil.make_archive(str(OUTPUT_ROOT), 'zip', OUTPUT_ROOT)
    print('Download:', archive)
